# Session 14: Distributions and sampling (Python lab)

You will calculate binomial and Gaussian probabilities, watch the law of large numbers and the central limit theorem happen, run an A/B test, and apply temperature and top-p to next-word probabilities.

**How to work.** In each step: read the idea, write a PREDICT guess, run the cell, READ the output, then CHANGE one thing in the experiment cell. Cells with `None  # your code here` are yours to fill. The automatic checks print OK or NOT YET. Run the cells from the top, in order. Every random step makes its own seeded generator, so your numbers match the slides.

---
## Step 0: Set up

numpy for numbers, matplotlib for pictures, math for combinations and the error function.

In [ ]:
import math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def Phi(z):
    """Area under the standard Gaussian curve to the left of z."""
    return 0.5 * (1 + math.erf(z / math.sqrt(2)))

print("ready")

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

---
## Step 1: Binomial: ten users, each rates helpful with probability 0.3

P(X = k) = (n choose k) p^k (1 - p)^(n - k). Illustrative numbers.

**PREDICT first.** Which count k is the most likely? Roughly what share of 100,000 simulated groups of ten will have exactly 3 helpful ratings?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: the formula from the slides, with math.comb for the number of ways. The pmf repeats it for every k from 0 to 10.
p3 = None  # your code here
pmf = None  # your code here: a list of 11 probabilities
rng = np.random.default_rng(14)
draws = rng.binomial(10, 0.3, size=100000)
sim_p3 = None  # your code here: the share of draws equal to 3
sim_mean = None  # your code here: the mean number of helpful ratings per group
check("P(X = 3) by formula", p3, 0.2668, 0.0001)
check("the pmf adds up to 1", None if pmf is None else sum(pmf), 1, 0.000001)
check("simulated share with exactly 3 is near the formula", sim_p3, 0.2668, 0.01)
check("simulated mean is near n p = 3", sim_mean, 3, 0.03)

**CHANGE one thing.** Run the experiment cell: P(at most 3 helpful ratings). Predict whether it is about 0.27, 0.65 or 0.95 first.

> What I saw: 

In [ ]:
print("P(X <= 3) =", round(sum(math.comb(10, k) * 0.3**k * 0.7**(10 - k) for k in range(4)), 4))

---
## Step 2: Gaussian: quiz scores with mean 60 and standard deviation 10

Illustrative numbers. The z-score is (x - mu) / sigma. `Phi(z)` from Step 0 gives the area to the left.

**PREDICT first.** About what share of learners score below 75? About what share score between 50 and 70?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: Phi(z) is the area to the left of z. For the area between two values, think of subtracting one area from another.
z75 = None  # your code here
p75 = None  # your code here
p_mid = None  # your code here
scores = np.random.default_rng(14).normal(60, 10, size=100000)
sim75 = None  # your code here
check("z-score of 75", z75, 1.5, 0.0001)
check("P(X < 75)", p75, 0.9332, 0.0001)
check("P(50 < X < 70)", p_mid, 0.6827, 0.0001)
check("simulated share below 75 is near the formula", sim75, 0.9332, 0.01)

**CHANGE one thing.** Run the experiment cell: the share within 2 standard deviations of the mean, between 40 and 80. Predict it first (the slides say about 95%).

> What I saw: 

In [ ]:
print("P(40 < X < 80) =", round(Phi(2) - Phi(-2), 4))

---
## Step 3: Law of large numbers and the central limit theorem

Roll a die many times and watch the running average settle. Then average 30 dice, 20,000 times, and see the bell shape. The standard deviation of one die is sqrt(35/12) = 1.7078.

**PREDICT first.** How close to 3.5 is the average after 10,000 rolls? What will the standard deviation of the averages of 30 dice be? (standard deviation of one die divided by the square root of 30)

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: the average after each roll is the total so far divided by the number of rolls so far. The standard error is on the slides.
rolls = np.random.default_rng(14).integers(1, 7, size=10000)
running = None  # your code here
rng = np.random.default_rng(14)
means = rng.integers(1, 7, size=(20000, 30)).mean(axis=1)
se_theory = None  # your code here
se_sim = None  # your code here: the standard deviation of means
first12 = None  # your code here: the first 12 of the 20,000 averages
mean_means = None  # your code here: the mean of all 20,000 averages
check("running average after 10,000 rolls is near 3.5", None if running is None else running[-1], 3.5, 0.05)
check("standard error from the formula", se_theory, 0.3118, 0.0001)
check("simulated standard deviation of the 30-dice averages", se_sim, 0.3118, 0.01)
check("mean of the 30-dice averages is near 3.5", mean_means, 3.5, 0.01)
check("mean of the first 12 averages (the workbook samples)", None if first12 is None else first12.mean(), 3.3972, 0.0005)
check("standard deviation of the first 12 averages", None if first12 is None else first12.std(), 0.3589, 0.0005)

In [ ]:
xs = np.linspace(2.2, 4.8, 200)
plt.figure(figsize=(6, 3))
plt.hist(means, bins=40, density=True, color="#E0BE36", edgecolor="black")
plt.plot(xs, np.exp(-0.5 * ((xs - 3.5) / 0.3118) ** 2) / (0.3118 * math.sqrt(2 * math.pi)), color="#AE2525")
plt.xlabel("average of 30 dice")
plt.savefig("s14_clt.png", dpi=80)
print("histogram saved")

**READ the output.** How well does the red Gaussian curve follow the histogram? A single die is flat. Why are the averages bell shaped?

> My answer: 

**CHANGE one thing.** Run the experiment cell: averages of 120 dice instead of 30. Predict the standard error first (four times the data).

> What I saw: 

In [ ]:
m120 = np.random.default_rng(14).integers(1, 7, size=(20000, 120)).mean(axis=1)
print(round(m120.std(), 4), round((35 / 12) ** 0.5 / math.sqrt(120), 4))

---
## Step 4: Confidence interval and A/B test

The 60 fixed rolls from Session 13, then illustrative A/B counts: A 288 of 2,400 rated helpful, B 336 of 2,400.

**PREDICT first.** Will the 95% interval from the 60 rolls contain the true mean 3.5? Will the A/B p-value be below 0.05?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
rolls60 = np.array([6, 6, 5, 6, 1, 5, 6, 2, 2, 1, 5, 6, 4, 4, 5, 1, 6, 6, 1, 6, 6, 2, 3, 5, 6, 1, 4, 3, 3, 5, 5, 3, 6, 4, 6, 1, 6, 5, 4, 3, 2, 2, 2, 5, 3, 6, 3, 4, 3, 5, 3, 6, 4, 1, 6, 4, 5, 6, 2, 1])
xa, na, xb, nb = 288, 2400, 336, 2400
print(rolls60.mean(), xa / na, xb / nb)

In [ ]:
# Hint: follow the steps of the A/B table on the slides. The test pools the two groups; the interval for the gap does not.
# se_diff is the standard error of the gap with no pooling: add p(1-p)/n for each group, then take the square root.
ci_lo, ci_hi = None, None  # your code here
se_diff = None  # your code here: the standard error of the gap, no pooling
z = None  # your code here
p_value = None  # your code here
gap_lo, gap_hi = None, None  # your code here
check("interval of the 60 rolls, lower end", ci_lo, 3.515, 0.001)
check("interval of the 60 rolls, upper end", ci_hi, 4.418, 0.001)
check("A/B test z statistic", z, 2.0601, 0.001)
check("A/B test two-sided p-value", p_value, 0.0394, 0.0005)
check("interval for the gap, lower end", gap_lo, 0.0010, 0.0002)
check("interval for the gap, upper end", gap_hi, 0.0390, 0.0002)

**READ the output.** The interval from the 60 rolls is just above 3.5. We know the true mean is 3.5. What does that tell you about a 95% interval? Is the A/B effect convincing, given the interval for the gap?

> My answer: 

**CHANGE one thing.** Run the experiment cell: the same rates with 4 times the users (9,600 each). Predict whether the p-value gets smaller or larger.

> What I saw: 

In [ ]:
na2 = nb2 = 9600
se2 = math.sqrt(0.13 * 0.87 * (1 / na2 + 1 / nb2))
z2 = 0.02 / se2
print(round(z2, 3), round(2 * (1 - Phi(z2)), 6))

---
## Step 5: Temperature on toy logits

Illustrative logits for five next words. p_i = exp(z_i / T) / sum of exp(z_j / T). Subtract the largest z / T first (stable softmax, Session 12).

**PREDICT first.** Which temperature gives tea the highest probability, 0.5, 1 or 2?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
words = ["tea", "coffee", "water", "milk", "soup"]
z = np.array([3.0, 2.6, 1.9, 1.2, 0.4])
# Hint: the stable softmax of Session 12, applied to the logits divided by T.
def softmax_t(z, T):
    return None  # your code here
p05, p1, p2 = softmax_t(z, 0.5), softmax_t(z, 1.0), softmax_t(z, 2.0)
check("P(tea) at T = 0.5", None if p05 is None else p05[0], 0.6278, 0.0005)
check("P(tea) at T = 1", None if p1 is None else p1[0], 0.4459, 0.0005)
check("P(tea) at T = 2", None if p2 is None else p2[0], 0.3252, 0.0005)
check("probabilities at T = 2 add up to 1", None if p2 is None else p2.sum(), 1, 0.000001)

**CHANGE one thing.** Run the experiment cell: T = 0.2. Predict P(tea) first. What does T near 0 do?

> What I saw: 

In [ ]:
zz = np.array([3.0, 2.6, 1.9, 1.2, 0.4])
s = zz / 0.2
e = np.exp(s - s.max())
print(np.round(e / e.sum(), 4))

---
## Step 6: Top-p (nucleus) sampling and drawing words

Keep the smallest set of the most likely words whose probabilities add up to at least p = 0.8, renormalise, then draw words with random numbers u (the slices idea).

**PREDICT first.** How many words are kept at p = 0.8? How often does tea appear in 8 draws at T = 0.5 compared with T = 2?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Hint: use the running total of the sorted probabilities to decide how many words reach 0.8. A draw is the first word whose running total reaches u.
order = np.argsort(-p1) if p1 is not None else None
k = None  # your code here
p_nuc = None  # your code here: keep the first k words, then divide by their sum
u = np.random.default_rng(19).random(8)
tea_counts = None  # your code here: for p1, p05 and p2, count how many of the 8 draws are "tea"
check("number of words kept at top-p 0.8", k, 3, 0)
check("renormalised P(tea)", None if p_nuc is None else p_nuc[0], 0.4992, 0.0005)
check("tea in 8 draws at T = 1, 0.5 and 2", tea_counts, [4, 5, 3], 0)

**READ the output.** Which words can never be drawn after the top-p cut? Why does the same question give different words with different random numbers u?

> My answer: 

**CHANGE one thing.** Run the experiment cell: top-p 0.95. Predict how many words are kept.

> What I saw: 

In [ ]:
cum = np.cumsum(softmax_t(np.array([3.0, 2.6, 1.9, 1.2, 0.4]), 1.0))
print(int(np.searchsorted(cum, 0.95)) + 1, "words kept")

---
## Step 7: Show the checks

Run this last cell. A learner who has filled every answer sees "All 27 checks pass".

In [ ]:
show_checks()